# Трійкове дерево пошуку (Ternary Search Tree)

**Трійкове дерево пошуку (TST)** — це структура даних для зберігання рядків, де кожен вузол містить один символ і три дочірні посилання:
- **left** — символи, менші за поточний
- **mid** — наступний символ рядка (якщо поточний збігається)
- **right** — символи, більші за поточний

TST поєднує переваги хеш-таблиць (швидкий пошук) і бінарних дерев пошуку (впорядкованість). Ефективний для автодоповнення, словників і spell-checkers.

# Теоретична база: Рядкові алгоритми та структури даних

---

## Постановка задач

Розглянемо п'ять фундаментальних задач на рядках. Маємо **N рядків**, кожен довжиною не більше **K**, над **алфавітом** заданого розміру (наприклад, латинські літери, `{0,1}`, або нуклеотиди ДНК).

### Задача 1 — Сортування рядків
Відсортувати n рядків лексикографічно. Наївний підхід (merge sort) дає **O(NK log N)** — кожне порівняння двох рядків коштує O(K). Мета — досягти **O(NK)**.

### Задача 2 — Множина рядків (Вставити / Знайти)
Підтримувати множину рядків з операціями: додати рядок і перевірити наявність рядка. Збалансоване BST дає **O(K log N)** на операцію.

### Задача 3 — Пошук за префіксом
Задачу 2 розширено: дано рядок **P** — знайти всі рядки в множині, для яких P є префіксом, та порахувати їх кількість.

### Задача 4 — Найдовша повторювана підстрока
Дано рядок **S** — знайти найдовшу підстроку, яка зустрічається щонайменше двічі (або R разів) у S.

### Задача 5 — Пошук підрядка
Дано рядок **S** та n запитів: для кожного запиту T — чи є T підрядком S, скільки разів зустрічається і на яких позиціях.

---

## Обмеження підходів на основі BST

Збалансоване двійкове дерево пошуку (AVL тощо) зберігає рядки і дозволяє їх порівнювати. Але кожне порівняння **двох рядків** коштує O(K), тому:

| Операція | BST |
|----------|-----|
| Сортування | O(NK log N) |
| Вставити / Знайти | O(K log N) |
| Пошук за префіксом | O(K log N) |

**Проблема:** множник K у кожній операції є зайвим. Ідеальна ціль — позбутися цього множника або замінити добуток на суму.

---

## Трійкове дерево пошуку (Ternary Search Tree, TST)

### Ключова ідея
Порівнювати не цілі рядки, а **по одному символу** за раз — тоді кожне порівняння коштує O(1).

### Структура вузла
Кожен вузол зберігає **один символ** і має **три дочірніх посилання**:
- **left** — рядки, чий поточний символ **менший** за символ вузла
- **right** — рядки, чий поточний символ **більший**
- **mid** — рядки, чий поточний символ **рівний** (перехід до наступного символу)

Переміщення вліво/вправо не просуває нас по рядку — ми залишаємось на тій самій позиції. Переміщення вниз через **mid** просуває нас до наступного символу.

### Складність операцій у TST
Нехай **H** — висота дерева, **K** — довжина рядка:
- Вниз через **mid** ми спускаємось щонайбільше K разів (по одному символу)
- Вліво/вправо — щонайбільше H разів

Тому час операції — **O(K + H)**.

У **збалансованому** TST (H = O(log N)):

| Операція | Збалансований TST |
|----------|-------------------|
| Вставити / Знайти | O(K + log N) |
| Сортування | O(K + log N) на рядок |
| Пошук за префіксом | O(K + log N) |

**O(K + log N)** суттєво краще за **O(K · log N)** при великих K.

> Балансування TST виконується через ротації (аналог AVL), але з урахуванням трьох дочірніх вузлів. Висота лишається O(log N).

> **Важлива перевага TST:** розмір алфавіту не впливає на складність — достатньо вміти порівнювати два символи за O(1). Тому TST підходить для сортування навіть векторів цілих чисел.

---

## Порівняльна таблиця структур

| Структура | Вставити / Знайти | Пам'ять | Умова |
|-----------|-------------|---------|-------|
| BST (збалансований) | O(K log N) | O(NK) | Будь-який алфавіт |
| TST (збалансований) | O(K + log N) | O(NK) | Будь-який алфавіт |

---


## 1. Клас вузла `TSTNode`

Кожен вузол зберігає один символ (`char`), прапорець кінця слова (`is_terminal`) та три посилання на дочірні вузли. Використання `__slots__` оптимізує пам'ять — Python не створює `__dict__` для кожного екземпляра.

In [13]:
class TSTNode:
    __slots__ = ("char", "is_terminal", "left", "mid", "right")

    def __init__(self, char: str) -> None:
        self.char = char
        self.is_terminal = False
        self.left = self.mid = self.right = None

## 2. Ініціалізація дерева `TernarySearchTree.__init__()`

Дерево зберігає посилання на кореневий вузол (`_root`) і лічильник унікальних слів (`_count`). Обидва поля ініціалізуються як "порожнє дерево".

In [14]:
class TernarySearchTree:

    def __init__(self) -> None:
        self._root: TSTNode | None = None
        self._count: int = 0

## 3. Вставка `insert` та `_insert`

### Алгоритм покроково

**Публічний метод `insert(value)`:**
1. Перевіряємо, що рядок `value` **не порожній** — якщо порожній, кидаємо `ValueError`.
2. Викликаємо `search(value)` — якщо слово **ще не існує** в дереві, збільшуємо лічильник `_count += 1`.
3. Запускаємо рекурсивну вставку `_insert(root, value, depth=0)` і оновлюємо корінь.

**Рекурсивний метод `_insert(node, value, depth)`:**
1. Беремо поточний символ: `ch = value[depth]`.
2. Якщо `node is None` — **створюємо новий вузол** `TSTNode(ch)` для цього символу.
3. Порівнюємо `ch` з `node.char`:
   - `ch < node.char` → символ **менший**: рекурсивно йдемо в `node.left`, `depth` не змінюється (ми ще на тій самій позиції рядка).
   - `ch > node.char` → символ **більший**: рекурсивно йдемо в `node.right`, `depth` не змінюється.
   - `ch == node.char` і `depth + 1 < len(value)` → символ **збігся**, але рядок ще не закінчився: йдемо в `node.mid` з `depth + 1` (переходимо до наступного символу рядка).
   - `ch == node.char` і це **останній символ** (`depth + 1 == len(value)`) → позначаємо `node.is_terminal = True` — слово повністю вставлено.
4. Повертаємо `node` (щоб батьківський виклик міг оновити посилання).


**Складність:** `O(L)`, де `L` — довжина рядка (проходимо кожен символ рівно один раз по `mid`-ланцюжку, плюс можливі порівняння по `left`/`right`).

In [15]:
def insert(self, value: str) -> None:
    if not value:
        raise ValueError("Значення не може бути порожнім рядком.")
    if not self.search(value):
        self._count += 1
    self._root = self._insert(self._root, value, 0)

def _insert(self, node: TSTNode | None, value: str, depth: int) -> TSTNode:
    ch = value[depth]
    if node is None:
        node = TSTNode(ch)
    if ch < node.char:
        node.left = self._insert(node.left, value, depth)
    elif ch > node.char:
        node.right = self._insert(node.right, value, depth)
    elif ch == node.char and depth + 1 < len(value):
        node.mid = self._insert(node.mid, value, depth + 1)
    else:
        node.is_terminal = True
    return node

## 4. Пошук `search` та `_find`

### Алгоритм покроково

**Публічний метод `search(value)`:**
1. Викликаємо `_find(root, value, depth=0)` — отримуємо вузол або `None`.
2. Повертаємо `True` тільки якщо вузол **знайдено** (`node is not None`) **і** `node.is_terminal == True`.
   - Вузол без `is_terminal` означає, що рядок є лише **префіксом** якогось слова, але не самостійним словом.

**Рекурсивний метод `_find(node, value, depth)`:**
1. Якщо `node is None` → вузол не існує → повертаємо `None` (слово не знайдено).
2. Беремо поточний символ: `ch = value[depth]`.
3. Порівнюємо `ch` з `node.char`:
   - `ch < node.char` → йдемо в `node.left` з тим самим `depth`.
   - `ch > node.char` → йдемо в `node.right` з тим самим `depth`.
   - `ch == node.char` і `depth + 1 == len(value)` → **останній символ збігся** → повертаємо поточний `node`.
   - `ch == node.char` і є ще символи → йдемо в `node.mid` з `depth + 1`.

**Складність:** `O(L)`, де `L` — довжина рядка.

In [16]:
def search(self, value: str) -> bool:
    node = self._find(self._root, value, 0)
    return node is not None and node.is_terminal

def _find(self, node: TSTNode | None, value: str, depth: int) -> TSTNode | None:
    if node is None:
        return None
    ch = value[depth]
    if ch < node.char:
        return self._find(node.left, value, depth)
    if ch > node.char:
        return self._find(node.right, value, depth)
    if depth + 1 == len(value):
        return node
    return self._find(node.mid, value, depth + 1)

## 5. Видалення `delete` та `_delete`

### Алгоритм покроково

**Публічний метод `delete(value)`:**
1. Якщо рядок **порожній** — одразу виходимо, нічого не робимо.
2. Запускаємо рекурсивне видалення `_delete(root, value, depth=0)`, яке повертає `(was_deleted, new_node)`.
3. Оновлюємо корінь дерева.
4. Якщо `was_deleted == True` — зменшуємо лічильник `_count -= 1`.

**Рекурсивний метод `_delete(node, value, depth)` → `(bool, node)`:**
1. Якщо `node is None` → слово не знайдено → повертаємо `(False, None)`.
2. Беремо поточний символ: `ch = value[depth]`.
3. **Навігація** (як у `_find`) — рекурсивно обробляємо потрібну гілку:
   - `ch < node.char` → рекурсія в `node.left`, `depth` не змінюється.
   - `ch > node.char` → рекурсія в `node.right`, `depth` не змінюється.
   - `ch == node.char` і є ще символи → рекурсія в `node.mid`, `depth + 1`.
   - `ch == node.char` і це останній символ:
     - Якщо `node.is_terminal == True` → **знімаємо прапорець** (`is_terminal = False`), `was_deleted = True`.
     - Якщо `node.is_terminal == False` → слова немає в дереві, `was_deleted = False`.
4. **Прунінг (очищення порожніх вузлів):** якщо `was_deleted == True` І вузол більше **не термінальний** І **не має `mid`-дітей** І **не має `left` і `right`** → вузол стає непотрібним → повертаємо `(True, None)` (видаляємо вузол).
5. В усіх інших випадках — повертаємо `(was_deleted, node)`.

**Складність:** `O(L)`, де `L` — довжина рядка.

In [17]:
def delete(self, value: str) -> None:
    if not value:
        return
    was_deleted, self._root = self._delete(self._root, value, 0)
    if was_deleted:
        self._count -= 1

def _delete(self, node: TSTNode | None, value: str, depth: int) -> tuple[bool, TSTNode | None]:
    if node is None:
        return False, None
    ch = value[depth]
    was_deleted = False
    if ch < node.char:
        was_deleted, node.left = self._delete(node.left, value, depth)
    elif ch > node.char:
        was_deleted, node.right = self._delete(node.right, value, depth)
    elif depth + 1 < len(value):
        was_deleted, node.mid = self._delete(node.mid, value, depth + 1)
    elif node.is_terminal:
        node.is_terminal, was_deleted = False, True
    if was_deleted and not node.is_terminal and node.mid is None:
        if node.left is None and node.right is None:
            return was_deleted, None
    return was_deleted, node

## 6. Симетричний обхід `inorder` та `_inorder`

### Алгоритм покроково

**Публічний метод `inorder()`:**
1. Створюємо порожній список `collected` для результатів.
2. Запускаємо рекурсивний обхід `_inorder(root, stack=[], collected)`.
3. Повертаємо `collected`.

**Рекурсивний метод `_inorder(node, stack, collected)`:**
1. Базовий випадок: якщо `node is None` → виходимо.
2. **Крок 1 — обходимо `left`:** рекурсивно викликаємо `_inorder(node.left, ...)`. `stack` не змінюється — `left` є бічною гілкою з символами **меншими** за поточний на тому самому рівні рядка.
3. **Крок 2 — фіксуємо поточний вузол:** додаємо `node.char` до `stack`.
4. **Крок 3 — обходимо `mid`:** рекурсивно викликаємо `_inorder(node.mid, ...)`. `mid` продовжує поточний префікс наступними символами.
5. **Крок 4 — перевіряємо термінальність:** якщо `node.is_terminal == True` → `stack` утворює повне слово → зберігаємо `"".join(stack)` у `collected`.
6. **Крок 5 — знімаємо символ:** `stack.pop()` — прибираємо `node.char` зі стеку.
7. **Крок 6 — обходимо `right`:** рекурсивно викликаємо `_inorder(node.right, ...)`. `right` містить символи **більші** за поточний на тому самому рівні.

> **Чому порядок `left → mid → WORD → right` НЕ дає строго лексикографічний результат?**  
> Якщо слово є префіксом іншого (`"cat"` / `"cats"`), воно з'явиться **після** своїх продовжень: `cats` іде перед `cat`. Строго лексикографічний порядок дає лише `preorder` (`left → WORD → mid → right`), де коротше слово фіксується до занурення в `mid`.

**Складність:** `O(N · L)`, де `N` — кількість слів, `L` — середня довжина слова.

In [ ]:
def _inorder(self, node, stack, collected):
    if node is None:
        return
    self._inorder(node.left, stack, collected)
    stack.append(node.char)
    self._inorder(node.mid, stack, collected)   
    if node.is_terminal:                        
        collected.append("".join(stack))
    stack.pop()
    self._inorder(node.right, stack, collected)

## 7. Прямий обхід `preorder` та `_preorder`

### Алгоритм покроково

**Публічний метод `preorder()`:**
1. Створюємо порожній список `collected`.
2. Запускаємо `_preorder(root, stack=[], collected)`.
3. Повертаємо `collected` — слова у порядку **першого відвідування** вузлів.

**Рекурсивний метод `_preorder(node, stack, collected)`:**
1. Базовий випадок: якщо `node is None` → виходимо.
2. **Крок 1 — фіксуємо поточний вузол:** додаємо `node.char` до `stack` **одразу** при вході у вузол.
3. **Крок 2 — перевіряємо термінальність:** якщо `node.is_terminal == True` → зберігаємо `"".join(stack)` у `collected`.
4. **Крок 3 — обходимо `left`:** рекурсія в `node.left` (символи менші на цьому рівні).
5. **Крок 4 — обходимо `mid`:** рекурсія в `node.mid` (наступні символи поточного префіксу).
6. **Крок 5 — обходимо `right`:** рекурсія в `node.right` (символи більші на цьому рівні).
7. **Крок 6 — знімаємо символ:** `stack.pop()` — прибираємо `node.char` **після всіх трьох рекурсій**.

> **Відмінність від inorder:** символ додається до `stack` **до** рекурсії в `left`/`right`, тому `left`-вузли отримують у стеку символ батька. Через це слова виходять **не** в лексикографічному, а в порядку обходу дерева — зручно для серіалізації або копіювання структури.

**Складність:** `O(N · L)`.

In [19]:
def preorder(self) -> list[str]:
    collected: list[str] = []
    self._preorder(self._root, [], collected)
    return collected

def _preorder(self, node: TSTNode | None, stack: list[str], collected: list[str]) -> None:
    if node is None:
        return
    self._preorder(node.left, stack, collected)
    stack.append(node.char)
    if node.is_terminal:
        collected.append("".join(stack))
    self._preorder(node.mid, stack, collected)
    stack.pop()
    self._preorder(node.right, stack, collected)

## 8. Зворотній обхід `postorder` та `_postorder`

### Алгоритм покроково

**Публічний метод `postorder()`:**
1. Створюємо порожній список `collected`.
2. Запускаємо `_postorder(root, stack=[], collected)`.
3. Повертаємо `collected` — слова у порядку **після обробки всіх дітей** вузла.

**Рекурсивний метод `_postorder(node, stack, collected)`:**
1. Базовий випадок: якщо `node is None` → виходимо.
2. **Крок 1 — обходимо `left`:** рекурсія в `node.left` **без додавання символу до стеку** — `left` є бічною гілкою і не продовжує поточний префікс.
3. **Крок 2 — входимо у вузол для `mid`:** `stack.append(node.char)` — додаємо символ.
4. **Крок 3 — обходимо `mid`:** рекурсія в `node.mid` (зі символом у стеку).
5. **Крок 4 — виходимо після `mid`:** `stack.pop()` — прибираємо символ.
6. **Крок 5 — обходимо `right`:** рекурсія в `node.right` **без символу** (знову бічна гілка).
7. **Крок 6 — повертаємось до вузла:** `stack.append(node.char)` — знову додаємо символ для перевірки самого вузла.
8. **Крок 7 — перевіряємо термінальність:** якщо `node.is_terminal == True` → зберігаємо `"".join(stack)` у `collected`.
9. **Крок 8 — фінальне очищення:** `stack.pop()`.

> **Чому подвійний `append/pop`?** `left` і `right` — це **бічні** гілки TST: вони не продовжують поточний префікс, а є альтернативними символами на тому самому рівні. Тому під час їх обходу символ поточного вузла не повинен бути у стеку. Але коли ми перевіряємо `mid` і саму термінальність — символ потрібен. Звідси два окремих `append/pop`.

**Складність:** `O(N · L)`.

In [20]:
def postorder(self) -> list[str]:
    collected: list[str] = []
    self._postorder(self._root, [], collected)
    return collected

def _postorder(self, node: TSTNode | None, stack: list[str], collected: list[str]) -> None:
    if node is None:
        return
    self._postorder(node.left, stack, collected)
    stack.append(node.char)
    self._postorder(node.mid, stack, collected)
    stack.pop()
    self._postorder(node.right, stack, collected)
    stack.append(node.char)
    if node.is_terminal:
        collected.append("".join(stack))
    stack.pop()

## 9. Мінімум `min` та максимум `max`

### Алгоритм покроково

**Публічні методи `min()` / `max()`:**
1. Якщо `_root is None` → дерево порожнє → кидаємо `ValueError`.
2. Викликаємо `_extreme(root, stack=[], prefer_left=True)` для мінімуму або `prefer_left=False` для максимуму.

**Рекурсивний метод `_extreme(node, stack, prefer_left)` → `str | None`:**
1. Базовий випадок: якщо `node is None` → повертаємо `None`.
2. Визначаємо **пріоритетну** та **вторинну** гілки:
   - `prefer_left=True` (мінімум): `primary = node.left`, `secondary = node.right`
   - `prefer_left=False` (максимум): `primary = node.right`, `secondary = node.left`
3. **Крок 1 — шукаємо в пріоритетній гілці:** `result = _extreme(primary, stack, prefer_left)`.
   - Якщо знайдено (`result is not None`) → **одразу повертаємо** його. Пріоритетна гілка завжди дає менший (або більший) результат.
4. **Крок 2 — пріоритетна гілка порожня, розглядаємо поточний вузол:**
   - `stack.append(node.char)` — додаємо поточний символ до префіксу.
   - Рекурсивно шукаємо `mid_result = _extreme(node.mid, stack, prefer_left)` — йдемо глибше по поточному префіксу.
   - Формуємо `value`: беремо `mid_result`, або якщо `node.is_terminal == True` — `"".join(stack)` (поточний вузол і є словом).
   - `stack.pop()` — прибираємо символ.
5. **Крок 3 — якщо нічого не знайдено** (ні в `mid`, ні термінальний) → шукаємо в `secondary`-гілці.
6. Повертаємо знайдене значення або `None`.

**Складність:** `O(L · log N)` у збалансованому дереві.

In [21]:
def min(self) -> str:
    if self._root is None:
        raise ValueError("Дерево порожнє.")
    return self._extreme(self._root, [], prefer_left=True)

def max(self) -> str:
    if self._root is None:
        raise ValueError("Дерево порожнє.")
    return self._extreme(self._root, [], prefer_left=False)

def _extreme(self, node: TSTNode | None, stack: list[str], prefer_left: bool) -> str | None:
    if node is None:
        return None
    primary, secondary = (node.left, node.right) if prefer_left else (node.right, node.left)
    result = self._extreme(primary, stack, prefer_left)
    if result is not None:
        return result
    stack.append(node.char)
    mid_result = self._extreme(node.mid, stack, prefer_left)
    value = mid_result or ("".join(stack) if node.is_terminal else None)
    stack.pop()
    return value if value is not None else self._extreme(secondary, stack, prefer_left)

## 10. Висота дерева `height`

### Алгоритм покроково

**Метод `height()`:**
1. Визначає вкладену функцію `_h(node)` і викликає її від кореня.

**Вкладена функція `_h(node)` → `int`:**
1. Базовий випадок: якщо `node is None` → висота дорівнює `0` (порожнє піддерево не займає рівнів).
2. Рекурсивно обчислюємо висоту **всіх трьох** дочірніх піддерев:
   - `h_left = _h(node.left)`
   - `h_mid = _h(node.mid)`
   - `h_right = _h(node.right)`
3. Повертаємо `1 + max(h_left, h_mid, h_right)` — поточний вузол додає один рівень над найглибшим із дітей.

> **Важливо:** висота TST — це **не** довжина найдовшого слова. Наприклад, слова `"abc"` і `"xyz"` можуть зберігатись у дереві висотою 6, якщо `'a'` і `'x'` ростуть у різні бічні гілки. Висота = максимальна кількість вузлів на шляху від кореня до листка з урахуванням **усіх трьох** гілок.

**Складність:** `O(N · L)` — відвідуємо кожен вузол рівно один раз.

In [22]:
def height(self) -> int:
    import builtins
    def _h(node):
        if node is None:
            return 0
        return 1 + builtins.max(_h(node.left), _h(node.mid), _h(node.right))
    return _h(self._root)

## 11. Допоміжні методи

| Метод | Опис |
|-------|------|
| `size()` | Повертає кількість унікальних слів у дереві |
| `is_empty()` | `True` якщо дерево не містить жодного слова |
| `contains(value)` | Псевдонім для `search` — перевіряє наявність слова |
| `clear()` | Повністю очищає дерево, скидає лічильник |

In [23]:
def size(self) -> int:
    return self._count

def is_empty(self) -> bool:
    return self._count == 0

def contains(self, value: str) -> bool:
    return self.search(value)

def clear(self) -> None:
    self._root = None
    self._count = 0

## 12. Повна збірка класу `TernarySearchTree`

Об'єднуємо всі методи в один клас для подальшого використання.

In [ ]:
class TSTNode:
    __slots__ = ("char", "is_terminal", "left", "mid", "right")

    def __init__(self, char: str) -> None:
        self.char = char
        self.is_terminal = False
        self.left = self.mid = self.right = None


class TernarySearchTree:

    def __init__(self) -> None:
        self._root: TSTNode | None = None
        self._count: int = 0

    def insert(self, value: str) -> None:
        if not value:
            raise ValueError("Значення не може бути порожнім рядком.")
        if not self.search(value):
            self._count += 1
        self._root = self._insert(self._root, value, 0)

    def _insert(self, node, value, depth):
        ch = value[depth]
        if node is None:
            node = TSTNode(ch)
        if ch < node.char:
            node.left = self._insert(node.left, value, depth)
        elif ch > node.char:
            node.right = self._insert(node.right, value, depth)
        elif ch == node.char and depth + 1 < len(value):
            node.mid = self._insert(node.mid, value, depth + 1)
        else:
            node.is_terminal = True
        return node

    def search(self, value: str) -> bool:
        node = self._find(self._root, value, 0)
        return node is not None and node.is_terminal

    def _find(self, node, value, depth):
        if node is None:
            return None
        ch = value[depth]
        if ch < node.char:
            return self._find(node.left, value, depth)
        if ch > node.char:
            return self._find(node.right, value, depth)
        if depth + 1 == len(value):
            return node
        return self._find(node.mid, value, depth + 1)

    def delete(self, value: str) -> None:
        if not value:
            return
        was_deleted, self._root = self._delete(self._root, value, 0)
        if was_deleted:
            self._count -= 1

    def _delete(self, node, value, depth):
        if node is None:
            return False, None
        ch = value[depth]
        was_deleted = False
        if ch < node.char:
            was_deleted, node.left = self._delete(node.left, value, depth)
        elif ch > node.char:
            was_deleted, node.right = self._delete(node.right, value, depth)
        elif depth + 1 < len(value):
            was_deleted, node.mid = self._delete(node.mid, value, depth + 1)
        elif node.is_terminal:
            node.is_terminal, was_deleted = False, True
        if was_deleted and not node.is_terminal and node.mid is None:
            if node.left is None and node.right is None:
                return was_deleted, None
        return was_deleted, node

    def inorder(self) -> list:
        collected = []
        self._inorder(self._root, [], collected)
        return collected

    def _inorder(self, node, stack, collected):
        if node is None:
            return
        self._inorder(node.left, stack, collected)
        stack.append(node.char)
        self._inorder(node.mid, stack, collected)   
        if node.is_terminal:                        
            collected.append("".join(stack))
        stack.pop()
        self._inorder(node.right, stack, collected)

    def preorder(self) -> list:
        collected = []
        self._preorder(self._root, [], collected)
        return collected

    def _preorder(self, node, stack, collected):
        if node is None:
            return
        self._preorder(node.left, stack, collected)
        stack.append(node.char)
        if node.is_terminal:
            collected.append("".join(stack))
        self._preorder(node.mid, stack, collected)
        stack.pop()
        self._preorder(node.right, stack, collected)

    def postorder(self) -> list:
        collected = []
        self._postorder(self._root, [], collected)
        return collected

    def _postorder(self, node, stack, collected):
        if node is None:
            return
        self._postorder(node.left, stack, collected)
        stack.append(node.char)
        self._postorder(node.mid, stack, collected)
        stack.pop()
        self._postorder(node.right, stack, collected)
        stack.append(node.char)
        if node.is_terminal:
            collected.append("".join(stack))
        stack.pop()

    def min(self) -> str:
        if self._root is None:
            raise ValueError("Дерево порожнє.")
        return self._extreme(self._root, [], prefer_left=True)

    def max(self) -> str:
        if self._root is None:
            raise ValueError("Дерево порожнє.")
        return self._extreme(self._root, [], prefer_left=False)

    def _extreme(self, node, stack, prefer_left):
        if node is None:
            return None
        primary, secondary = (node.left, node.right) if prefer_left else (node.right, node.left)
        result = self._extreme(primary, stack, prefer_left)
        if result is not None:
            return result
        stack.append(node.char)
        mid_result = self._extreme(node.mid, stack, prefer_left)
        value = mid_result or ("".join(stack) if node.is_terminal else None)
        stack.pop()
        return value if value is not None else self._extreme(secondary, stack, prefer_left)

    def height(self) -> int:
        import builtins
        def _h(node):
            if node is None:
                return 0
            return 1 + builtins.max(_h(node.left), _h(node.mid), _h(node.right))
        return _h(self._root)

    def size(self) -> int:
        return self._count

    def is_empty(self) -> bool:
        return self._count == 0

    def contains(self, value: str) -> bool:
        return self.search(value)

    def clear(self) -> None:
        self._root = None
        self._count = 0

In [25]:
# ── 1. TSTNode ────────────────────────────────────────
node = TSTNode('c')
print("char:", node.char, "| is_terminal:", node.is_terminal, "| left:", node.left, "| mid:", node.mid, "| right:", node.right)


char: c | is_terminal: False | left: None | mid: None | right: None


In [26]:
# ── 2. __init__ ───────────────────────────────────────
tst = TernarySearchTree()
print("_root:", tst._root, "| _count:", tst._count)


_root: None | _count: 0


In [27]:
# ── 3. insert ─────────────────────────────────────────
tst1 = TernarySearchTree()
tst1.insert("cat")
tst1.insert("car")
tst1.insert("bat")
tst1.insert("cat")   # повторна вставка — лічильник не змінюється
print("size після 4 викликів (3 унікальних):", tst1.size())  # 3

try:
    tst1.insert("")   # порожній рядок → ValueError
except ValueError as e:
    print("Порожній рядок:", e)


size після 4 викликів (3 унікальних): 3
Порожній рядок: Значення не може бути порожнім рядком.


In [28]:
# ── 4. search ─────────────────────────────────────────
print("'cat':", tst1.search("cat"))   # True
print("'ca' :", tst1.search("ca"))    # False — лише префікс
print("'dog':", tst1.search("dog"))   # False — відсутнє
print("'bat':", tst1.search("bat"))   # True


'cat': True
'ca' : False
'dog': False
'bat': True


In [29]:
# ── 5. delete ─────────────────────────────────────────
tst2 = TernarySearchTree()
for w in ["cat", "car", "bat", "cats"]:
    tst2.insert(w)
print("До видалення:", tst2.inorder())        # ['bat', 'car', 'cat', 'cats']
tst2.delete("cat")
print("Після delete('cat'):", tst2.inorder()) # ['bat', 'car', 'cats']
print("'cats' ще є:", tst2.search("cats"))    # True — не зачеплено
tst2.delete("dog")                            # відсутнє — ігнорується
print("size:", tst2.size())                   # 3


До видалення: ['bat', 'car', 'cats', 'cat']
Після delete('cat'): ['bat', 'car', 'cats']
'cats' ще є: True
size: 3


In [30]:
# ── 6. inorder ────────────────────────────────────────
tst3 = TernarySearchTree()
for w in ["dog", "cat", "bat", "car", "ant"]:
    tst3.insert(w)
print("inorder:", tst3.inorder())   # ['ant', 'bat', 'car', 'cat', 'dog']


inorder: ['ant', 'bat', 'car', 'cat', 'dog']


In [31]:
# ── 7. preorder ───────────────────────────────────────
print("preorder:", tst3.preorder()) # ['ant', 'bat', 'car', 'cat', 'dog']


preorder: ['ant', 'bat', 'car', 'cat', 'dog']


In [32]:
# ── 8. postorder ──────────────────────────────────────
print("postorder:", tst3.postorder()) # ['ant', 'bat', 'car', 'cat', 'dog']


postorder: ['ant', 'bat', 'car', 'cat', 'dog']


In [33]:
# ── 9. min / max ──────────────────────────────────────
tst4 = TernarySearchTree()
for w in ["mango", "apple", "banana", "cherry", "avocado"]:
    tst4.insert(w)
print("min:", tst4.min())   # apple
print("max:", tst4.max())   # mango

try:
    TernarySearchTree().min()
except ValueError as e:
    print("Порожнє дерево →", e) # порожнє дерево → ValueError


min: apple
max: mango
Порожнє дерево → Дерево порожнє.


In [34]:
# ── 10. height ────────────────────────────────────────
tst5 = TernarySearchTree()
print("Порожнє:", tst5.height())   # 0
tst5.insert("a")
print("Після 'a':", tst5.height()) # 1
for w in ["cat", "car", "bat", "cats", "category"]:
    tst5.insert(w)
print("Після кількох слів:", tst5.height()) #10


Порожнє: 0
Після 'a': 1
Після кількох слів: 10


In [35]:
# ── 11. Допоміжні методи ──────────────────────────────
tst6 = TernarySearchTree()
print("is_empty:", tst6.is_empty())                      # True
print("size:", tst6.size())                              # 0
for w in ["alpha", "beta", "gamma"]:
    tst6.insert(w)
print("is_empty:", tst6.is_empty())                      # False
print("size:", tst6.size())                              # 3
print("contains('beta'):", tst6.contains("beta"))        # True
print("contains('delta'):", tst6.contains("delta"))      # False
tst6.clear()
print("Після clear — size:", tst6.size())                # 0
print("Після clear — is_empty:", tst6.is_empty())        # True


is_empty: True
size: 0
is_empty: False
size: 3
contains('beta'): True
contains('delta'): False
Після clear — size: 0
Після clear — is_empty: True
